============================================
# PROJETO DATA MAJOR - ETAPA 03: LOAD (CARGA)
Responsável: Lucas de Siqueira Cavalcanti (Load)

=============================================




In [1]:
from google.colab import drive
import os
from pyspark.sql import SparkSession
import sqlite3
import pandas as pd
import glob

Montagem do Drive

In [2]:
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/Topicos_BD'
path_processed = os.path.join(base_path, 'processed', 'sinan_dengue_processed')
db_path   = os.path.join(base_path, 'database', 'dengue_db.db')

os.makedirs(os.path.dirname(db_path), exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


inicialização da Sessão Spark

In [3]:
spark = SparkSession.builder \
    .appName('DataMajor_Load') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

Leitura do Parquet Processado no Transform

In [4]:
df_final = spark.read.parquet(path_processed)

Registro de View para SQL Puro
Para usar SQL para validar os dados antes da carga final

In [5]:
df_final.createOrReplaceTempView("tb_dengue_final")

In [6]:
spark.sql("""
    SELECT HOSPITALIZ, COUNT(*) as total_casos
    FROM tb_dengue_final
    GROUP BY HOSPITALIZ
""").show()

+----------+-----------+
|HOSPITALIZ|total_casos|
+----------+-----------+
|         1|     306550|
|         0|    6594826|
+----------+-----------+



In [7]:
padrao_busca = os.path.join(path_processed, '**', '*.parquet')
arquivos_parquet = glob.glob(padrao_busca, recursive=True)

tamanho_mb = sum(os.path.getsize(f) for f in arquivos_parquet) / 1e6

print(f"Tamanho total: {tamanho_mb:.2f} MB")

Tamanho total: 23.34 MB


Carga em Banco de Dados - SQLite

In [8]:
conn = sqlite3.connect(db_path)

print("Convertendo amostra para Pandas...")
# Re-reading the DataFrame to ensure Spark has a fresh view of the files
df_final_reloaded = spark.read.parquet(path_processed)
df_amostra = df_final_reloaded.limit(100000).toPandas()

print("Inserindo dados na tabela 'dengue_hospitalizacao'...")
df_amostra.to_sql('dengue_hospitalizacao', conn, if_exists='replace', index=False)

conn.close()

print(f"Arquivo de Banco de Dados (.db) disponível em: {db_path}")
print("Você pode baixar este arquivo e abrir no DBeaver.")

Convertendo amostra para Pandas...
Inserindo dados na tabela 'dengue_hospitalizacao'...
Arquivo de Banco de Dados (.db) disponível em: /content/drive/MyDrive/Topicos_BD/database/dengue_db.db
Você pode baixar este arquivo e abrir no DBeaver.


In [9]:
# spark.stop()